
# Multi-Timeframe Microstructure Mining, Backtest, and Meta-Labeling
Revisi untuk pipeline **M1 source -> M5 / M15 / H1 research**.

Default mode:
- **H1** = context
- **M15** = primary event + baseline execution
- **M5** = optional refinement
- **SL** = structural swing
- **TP** = 2R
- **timeout** = 16 bar untuk mode M15
- **spread** = 0


In [ ]:

import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:

CONFIG = {
    "data_path": "YOUR_XAUUSD_M1.csv",
    "source_tf": "M1",
    "resample_label": "right",
    "resample_closed": "right",
    "context_tf": "H1",
    "signal_tf": "M15",
    "execution_tf": "M15",
    "use_m5_refinement": False,
    "m5_refinement_window_bars": 3,
    "spread_points": 0.0,
    "tp_r_multiple": 2.0,
    "swing_lookback": 3,
    "rolling_level_n": 20,
    "timeout_bars_map": {"M5": 30, "M15": 16, "H1": 8},
    "train_ratio": 0.60,
    "val_ratio": 0.20,
    "output_dir": "outputs_mtf_microstructure",
}
CONFIG["timeout_bars"] = CONFIG["timeout_bars_map"][CONFIG["signal_tf"]]
RULE_MAP = {"M5": "5min", "M15": "15min", "H1": "1h"}
CONFIG


In [ ]:

def load_ohlcv(path: str) -> pd.DataFrame:
    names = ["date", "time", "open", "high", "low", "close", "volume"]
    df = pd.read_csv(path, names=names, skiprows=1)
    ts = pd.to_datetime(df["date"] + " " + df["time"], format="%Y.%m.%d %H:%M", utc=True)
    df = df.drop(columns=["date", "time"])
    df.index = ts
    df.index.name = "timestamp"
    df = df.sort_index()
    df = df[~df.index.duplicated(keep="first")]
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def validate_m1(df: pd.DataFrame) -> pd.Series:
    diffs = df.index.to_series().diff().dropna()
    expected = pd.Timedelta(minutes=1)
    return pd.Series({
        "rows": len(df),
        "start": str(df.index.min()),
        "end": str(df.index.max()),
        "duplicate_index": int(df.index.duplicated().sum()),
        "missing_1m_gaps": int((diffs > expected).sum()),
        "nonpositive_price_rows": int(((df[["open","high","low","close"]] <= 0).any(axis=1)).sum()),
        "null_rows": int(df.isna().any(axis=1).sum()),
    })


def resample_ohlcv(df: pd.DataFrame, rule: str, label="right", closed="right") -> pd.DataFrame:
    out = pd.DataFrame({
        "open": df["open"].resample(rule, label=label, closed=closed).first(),
        "high": df["high"].resample(rule, label=label, closed=closed).max(),
        "low": df["low"].resample(rule, label=label, closed=closed).min(),
        "close": df["close"].resample(rule, label=label, closed=closed).last(),
        "volume": df["volume"].resample(rule, label=label, closed=closed).sum(),
    }).dropna()
    return out


In [ ]:

def add_core_features(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["range"] = x["high"] - x["low"]
    x["body"] = x["close"] - x["open"]
    x["abs_body"] = x["body"].abs()
    x["upper_wick"] = x["high"] - x[["open", "close"]].max(axis=1)
    x["lower_wick"] = x[["open", "close"]].min(axis=1) - x["low"]
    x["body_ratio"] = np.where(x["range"] > 0, x["abs_body"] / x["range"], 0.0)
    x["close_position"] = np.where(x["range"] > 0, (x["close"] - x["low"]) / x["range"], 0.5)

    for n in [1, 2, 3, 5, 10]:
        x[f"ret_{n}"] = x["close"].pct_change(n)

    for n in [5, 10, 20]:
        x[f"vol_{n}"] = x["close"].pct_change().rolling(n).std()
        x[f"range_mean_{n}"] = x["range"].rolling(n).mean()

    prev_close = x["close"].shift(1)
    tr = pd.concat([
        x["high"] - x["low"],
        (x["high"] - prev_close).abs(),
        (x["low"] - prev_close).abs(),
    ], axis=1).max(axis=1)
    x["atr_14"] = tr.rolling(14).mean()

    for span in [20, 50]:
        x[f"ema_{span}"] = x["close"].ewm(span=span, adjust=False).mean()
        x[f"ema_{span}_slope"] = x[f"ema_{span}"].diff()

    ma = x["close"].rolling(20).mean()
    std = x["close"].rolling(20).std()
    x["bb_mid"] = ma
    x["bb_upper"] = ma + 2.0 * std
    x["bb_lower"] = ma - 2.0 * std
    x["bb_width"] = x["bb_upper"] - x["bb_lower"]
    x["bb_width_atr"] = np.where(x["atr_14"] > 0, x["bb_width"] / x["atr_14"], np.nan)

    n = CONFIG["rolling_level_n"]
    x["rolling_high_n"] = x["high"].shift(1).rolling(n).max()
    x["rolling_low_n"] = x["low"].shift(1).rolling(n).min()
    x["dist_to_roll_high"] = x["close"] - x["rolling_high_n"]
    x["dist_to_roll_low"] = x["close"] - x["rolling_low_n"]

    up = (x["close"] > x["close"].shift(1)).astype(int)
    dn = (x["close"] < x["close"].shift(1)).astype(int)
    x["up_streak"] = up.groupby((up != up.shift()).cumsum()).cumsum()
    x["down_streak"] = dn.groupby((dn != dn.shift()).cumsum()).cumsum()

    net = x["close"].diff(10).abs()
    gross = x["close"].diff().abs().rolling(10).sum()
    x["eff_10"] = np.where(gross > 0, net / gross, np.nan)

    x["hour"] = x.index.hour
    x["weekday"] = x.index.weekday
    x["session"] = np.select(
        [
            (x["hour"] >= 0) & (x["hour"] < 7),
            (x["hour"] >= 7) & (x["hour"] < 13),
            (x["hour"] >= 13) & (x["hour"] < 17),
            (x["hour"] >= 17) & (x["hour"] < 24),
        ],
        ["asia", "london", "ny_overlap", "ny"],
        default="other",
    )
    return x


In [ ]:

def add_h1_regime(h1: pd.DataFrame) -> pd.DataFrame:
    x = h1.copy()
    bull = (x["close"] > x["ema_50"]) & (x["ema_50_slope"] > 0)
    bear = (x["close"] < x["ema_50"]) & (x["ema_50_slope"] < 0)
    x["h1_regime"] = np.select([bull, bear], [1, -1], default=0)
    return x


def map_higher_tf_context(signal_df: pd.DataFrame, higher_df: pd.DataFrame, cols: list) -> pd.DataFrame:
    return pd.merge_asof(
        signal_df.sort_index(),
        higher_df[cols].sort_index(),
        left_index=True,
        right_index=True,
        direction="backward",
        allow_exact_matches=True,
    )


def find_swings(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    x["swing_high"] = (
        (x["high"] > x["high"].shift(1)) &
        (x["high"] > x["high"].shift(2)) &
        (x["high"] >= x["high"].shift(-1)) &
        (x["high"] >= x["high"].shift(-2))
    )
    x["swing_low"] = (
        (x["low"] < x["low"].shift(1)) &
        (x["low"] < x["low"].shift(2)) &
        (x["low"] <= x["low"].shift(-1)) &
        (x["low"] <= x["low"].shift(-2))
    )
    x["last_swing_high"] = np.where(x["swing_high"], x["high"], np.nan)
    x["last_swing_low"] = np.where(x["swing_low"], x["low"], np.nan)
    x["last_swing_high"] = pd.Series(x["last_swing_high"], index=x.index).ffill()
    x["last_swing_low"] = pd.Series(x["last_swing_low"], index=x.index).ffill()
    return x


In [ ]:

def detect_events_m15(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    long_mom = (x["body_ratio"] > 0.6) & (x["close"] > x["rolling_high_n"])
    short_mom = (x["body_ratio"] > 0.6) & (x["close"] < x["rolling_low_n"])

    long_sweep = (x["low"] < x["rolling_low_n"]) & (x["close"] > x["rolling_low_n"])
    short_sweep = (x["high"] > x["rolling_high_n"]) & (x["close"] < x["rolling_high_n"])

    bb_q_low = x["bb_width_atr"].rolling(200).quantile(0.25)
    range_q_hi = x["range"].rolling(200).quantile(0.75)
    long_comp = (x["bb_width_atr"] < bb_q_low) & (x["body"] > 0) & (x["range"] > range_q_hi)
    short_comp = (x["bb_width_atr"] < bb_q_low) & (x["body"] < 0) & (x["range"] > range_q_hi)

    long_rej = (x["lower_wick"] > x["abs_body"] * 1.5) & (x["close_position"] > 0.6) & (x["low"] <= x["rolling_low_n"])
    short_rej = (x["upper_wick"] > x["abs_body"] * 1.5) & (x["close_position"] < 0.4) & (x["high"] >= x["rolling_high_n"])

    x["event_name"] = np.nan
    x["direction"] = 0
    event_map = [
        ("momentum_break_long", long_mom, 1),
        ("momentum_break_short", short_mom, -1),
        ("sweep_reclaim_long", long_sweep, 1),
        ("sweep_reclaim_short", short_sweep, -1),
        ("compression_expansion_long", long_comp, 1),
        ("compression_expansion_short", short_comp, -1),
        ("rejection_long", long_rej, 1),
        ("rejection_short", short_rej, -1),
    ]
    for name, mask, d in event_map:
        idx = mask & x["event_name"].isna()
        x.loc[idx, "event_name"] = name
        x.loc[idx, "direction"] = d

    x["has_event"] = x["direction"] != 0
    return x


def apply_h1_filter(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    pass_long = (x["direction"] == 1) & (x["h1_regime"] == 1)
    pass_short = (x["direction"] == -1) & (x["h1_regime"] == -1)
    x["passes_h1_filter"] = pass_long | pass_short
    return x


In [ ]:

def optional_m5_refinement(m5: pd.DataFrame, signal_time: pd.Timestamp, direction: int, window_bars: int = 3) -> bool:
    loc = m5.index.searchsorted(signal_time)
    window = m5.iloc[loc+1:loc+1+window_bars].copy()
    if len(window) == 0:
        return False
    if direction == 1:
        return bool(((window["close"] > window["ema_20"]).any()) and ((window["body"] > 0).any()))
    if direction == -1:
        return bool(((window["close"] < window["ema_20"]).any()) and ((window["body"] < 0).any()))
    return False


def build_signal_table(m15: pd.DataFrame, m5: pd.DataFrame | None = None, use_m5_refinement: bool = False) -> pd.DataFrame:
    sig = m15[(m15["has_event"]) & (m15["passes_h1_filter"])].copy()
    sig["m5_refined"] = True
    if use_m5_refinement and m5 is not None:
        sig["m5_refined"] = [
            optional_m5_refinement(m5, ts, int(row["direction"]), CONFIG["m5_refinement_window_bars"])
            for ts, row in sig.iterrows()
        ]
        sig = sig[sig["m5_refined"]].copy()
    return sig


In [ ]:

def backtest_signals(df: pd.DataFrame, signals: pd.DataFrame, timeout_bars: int, tp_r_multiple: float = 2.0) -> pd.DataFrame:
    trades = []
    indexer = {ts: i for i, ts in enumerate(df.index)}

    for ts, row in signals.iterrows():
        i = indexer.get(ts)
        if i is None or i + 1 >= len(df):
            continue
        direction = int(row["direction"])
        entry_bar = i + 1
        entry_time = df.index[entry_bar]
        entry_price = float(df.iloc[entry_bar]["open"])

        if direction == 1:
            sl = float(row["last_swing_low"])
            if not np.isfinite(sl) or sl >= entry_price:
                continue
            risk = entry_price - sl
            tp = entry_price + tp_r_multiple * risk
        else:
            sl = float(row["last_swing_high"])
            if not np.isfinite(sl) or sl <= entry_price:
                continue
            risk = sl - entry_price
            tp = entry_price - tp_r_multiple * risk
        if risk <= 0:
            continue

        exit_price = np.nan
        exit_time = pd.NaT
        exit_reason = "timeout"
        holding_bars = 0

        for j in range(entry_bar, min(entry_bar + timeout_bars, len(df) - 1) + 1):
            bar = df.iloc[j]
            o, h, l = float(bar["open"]), float(bar["high"]), float(bar["low"])
            holding_bars = j - entry_bar + 1

            if direction == 1 and o <= sl:
                exit_price = o
                exit_reason = "gap_sl"
                exit_time = df.index[j]
                break
            if direction == -1 and o >= sl:
                exit_price = o
                exit_reason = "gap_sl"
                exit_time = df.index[j]
                break

            if direction == 1:
                hit_sl = l <= sl
                hit_tp = h >= tp
                if hit_sl and hit_tp:
                    exit_price = sl
                    exit_reason = "sl_first_tie"
                    exit_time = df.index[j]
                    break
                if hit_sl:
                    exit_price = sl
                    exit_reason = "sl"
                    exit_time = df.index[j]
                    break
                if hit_tp:
                    exit_price = tp
                    exit_reason = "tp"
                    exit_time = df.index[j]
                    break
            else:
                hit_sl = h >= sl
                hit_tp = l <= tp
                if hit_sl and hit_tp:
                    exit_price = sl
                    exit_reason = "sl_first_tie"
                    exit_time = df.index[j]
                    break
                if hit_sl:
                    exit_price = sl
                    exit_reason = "sl"
                    exit_time = df.index[j]
                    break
                if hit_tp:
                    exit_price = tp
                    exit_reason = "tp"
                    exit_time = df.index[j]
                    break

        if pd.isna(exit_time):
            j = min(entry_bar + timeout_bars, len(df)-1)
            exit_time = df.index[j]
            exit_price = float(df.iloc[j]["close"])

        r_mult = (exit_price - entry_price) / risk if direction == 1 else (entry_price - exit_price) / risk

        trades.append({
            "signal_time": ts,
            "entry_time": entry_time,
            "exit_time": exit_time,
            "event_name": row["event_name"],
            "direction": direction,
            "entry_price": entry_price,
            "sl_price": sl,
            "tp_price": tp,
            "exit_price": exit_price,
            "exit_reason": exit_reason,
            "risk": risk,
            "r_mult": r_mult,
            "holding_bars": holding_bars,
            "session": row["session"],
            "h1_regime": row["h1_regime"],
        })

    trades = pd.DataFrame(trades)
    if len(trades):
        trades["cum_r"] = trades["r_mult"].cumsum()
    return trades


def compute_metrics(trades: pd.DataFrame) -> dict:
    if trades.empty:
        return {
            "total_trades": 0,
            "win_rate": np.nan,
            "profit_factor": np.nan,
            "expectancy_r": np.nan,
            "avg_r": np.nan,
            "median_r": np.nan,
            "sum_r": 0.0,
            "max_drawdown_r": np.nan,
            "max_losing_streak": 0,
            "avg_holding_bars": np.nan,
        }
    gross_profit = trades.loc[trades["r_mult"] > 0, "r_mult"].sum()
    gross_loss = -trades.loc[trades["r_mult"] < 0, "r_mult"].sum()
    pf = gross_profit / gross_loss if gross_loss > 0 else np.inf
    eq = trades["r_mult"].cumsum()
    dd = eq - eq.cummax()
    max_losing_streak = 0
    cur = 0
    for x in trades["r_mult"]:
        if x < 0:
            cur += 1
            max_losing_streak = max(max_losing_streak, cur)
        else:
            cur = 0
    return {
        "total_trades": int(len(trades)),
        "win_rate": float((trades["r_mult"] > 0).mean()),
        "profit_factor": float(pf) if np.isfinite(pf) else None,
        "expectancy_r": float(trades["r_mult"].mean()),
        "avg_r": float(trades["r_mult"].mean()),
        "median_r": float(trades["r_mult"].median()),
        "sum_r": float(trades["r_mult"].sum()),
        "max_drawdown_r": float(dd.min()),
        "max_losing_streak": int(max_losing_streak),
        "avg_holding_bars": float(trades["holding_bars"].mean()),
    }


In [ ]:

def add_split_labels(df: pd.DataFrame, train_ratio: float, val_ratio: float) -> pd.Series:
    n = len(df)
    i1 = int(n * train_ratio)
    i2 = int(n * (train_ratio + val_ratio))
    split = pd.Series(index=df.index, dtype="object")
    split.iloc[:i1] = "train"
    split.iloc[i1:i2] = "validation"
    split.iloc[i2:] = "test"
    return split


def summarize_by_group(trades: pd.DataFrame, group_col: str) -> pd.DataFrame:
    rows = []
    for g, sub in trades.groupby(group_col):
        m = compute_metrics(sub)
        m[group_col] = g
        rows.append(m)
    return pd.DataFrame(rows)


In [ ]:

# === DATA PREP ===
# 1) Set CONFIG["data_path"] ke file M1 kamu
# 2) Jalankan cell ini

# m1 = load_ohlcv(CONFIG["data_path"])
# qc_report = validate_m1(m1)
# display(qc_report)

# m5 = add_core_features(resample_ohlcv(m1, RULE_MAP["M5"], CONFIG["resample_label"], CONFIG["resample_closed"]))
# m15 = add_core_features(resample_ohlcv(m1, RULE_MAP["M15"], CONFIG["resample_label"], CONFIG["resample_closed"]))
# h1 = add_core_features(resample_ohlcv(m1, RULE_MAP["H1"], CONFIG["resample_label"], CONFIG["resample_closed"]))

# h1 = add_h1_regime(h1)
# m5 = find_swings(m5)
# m15 = find_swings(m15)
# m15 = detect_events_m15(m15)

# m15 = map_higher_tf_context(m15, h1, ["h1_regime", "ema_50_slope", "bb_width_atr"])
# m15 = m15.rename(columns={"ema_50_slope": "h1_ema_50_slope", "bb_width_atr": "h1_bb_width_atr"})
# m15 = apply_h1_filter(m15)
# m15["split"] = add_split_labels(m15, CONFIG["train_ratio"], CONFIG["val_ratio"])

# signals = build_signal_table(m15, m5=m5, use_m5_refinement=CONFIG["use_m5_refinement"])
# signals["split"] = m15.loc[signals.index, "split"]
# print("M5 rows:", len(m5), "M15 rows:", len(m15), "H1 rows:", len(h1), "Signals:", len(signals))
# signals[["event_name", "direction", "session", "h1_regime", "split"]].head()


In [ ]:

# === BACKTEST ===
# trades = backtest_signals(m15, signals, timeout_bars=CONFIG["timeout_bars"], tp_r_multiple=CONFIG["tp_r_multiple"])
# if len(trades):
#     trades["split"] = pd.Series(trades["signal_time"]).map(m15["split"]).values
#     trades["year"] = pd.to_datetime(trades["entry_time"]).dt.year

# overall_metrics = compute_metrics(trades)
# print(json.dumps(overall_metrics, indent=2))
# display(summarize_by_group(trades, "split"))
# display(summarize_by_group(trades, "event_name"))
# display(summarize_by_group(trades, "session"))
# display(summarize_by_group(trades, "year"))


In [ ]:

# === EQUITY CURVE ===
# if len(trades):
#     eq = trades.set_index("exit_time")["r_mult"].cumsum()
#     plt.figure(figsize=(12, 5))
#     plt.plot(eq.index, eq.values)
#     plt.title("Equity Curve in R")
#     plt.xlabel("Exit Time")
#     plt.ylabel("Cumulative R")
#     plt.grid(True, alpha=0.3)
#     plt.show()


In [ ]:

def build_meta_dataset(signals: pd.DataFrame, trades: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    y = trades[["signal_time", "r_mult"]].copy()
    y["label"] = (y["r_mult"] > 0).astype(int)
    ds = signals[feature_cols].copy()
    ds = ds.merge(y[["signal_time", "label", "r_mult"]], left_index=True, right_on="signal_time", how="inner")
    ds = ds.set_index("signal_time").sort_index()
    return ds


def train_meta_models(ds: pd.DataFrame, split_series: pd.Series, feature_cols: list) -> dict:
    ds = ds.copy()
    ds["split"] = split_series.reindex(ds.index)
    train = ds[ds["split"] == "train"].copy()
    val = ds[ds["split"] == "validation"].copy()
    test = ds[ds["split"] == "test"].copy()

    X_train, y_train = train[feature_cols], train["label"]
    X_val, y_val = val[feature_cols], val["label"]
    X_test, y_test = test[feature_cols], test["label"]

    lr = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=200, class_weight="balanced")),
    ])
    gb = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=200)),
    ])

    out = {}
    for name, model in [("logreg", lr), ("hgb", gb)]:
        model.fit(X_train, y_train)
        p_val = model.predict_proba(X_val)[:, 1]
        p_test = model.predict_proba(X_test)[:, 1]
        out[name] = {
            "model": model,
            "val_auc": float(roc_auc_score(y_val, p_val)) if len(np.unique(y_val)) > 1 else np.nan,
            "test_auc": float(roc_auc_score(y_test, p_test)) if len(np.unique(y_test)) > 1 else np.nan,
            "test_probs": pd.Series(p_test, index=X_test.index),
        }
    return out


def threshold_filter_analysis(trades: pd.DataFrame, probs: pd.Series, thresholds=(0.5, 0.55, 0.6, 0.65, 0.7)):
    rows = []
    t = trades.set_index("signal_time")
    for th in thresholds:
        keep = probs[probs >= th].index
        sub = t.loc[t.index.intersection(keep)].reset_index()
        m = compute_metrics(sub)
        m["threshold"] = th
        rows.append(m)
    return pd.DataFrame(rows)


In [ ]:

# === META-LABELING ===
# feature_cols = [
#     "body_ratio", "close_position", "ret_1", "ret_3", "ret_5",
#     "vol_5", "vol_10", "atr_14", "bb_width_atr",
#     "dist_to_roll_high", "dist_to_roll_low",
#     "up_streak", "down_streak", "eff_10",
#     "hour", "weekday", "h1_regime", "h1_ema_50_slope", "h1_bb_width_atr",
# ]
# feature_cols = [c for c in feature_cols if c in signals.columns]
# meta_ds = build_meta_dataset(signals, trades, feature_cols)
# results = train_meta_models(meta_ds, m15["split"], feature_cols)
# for name, res in results.items():
#     print(name, "val_auc=", res["val_auc"], "test_auc=", res["test_auc"])
# best_name = max(results, key=lambda k: np.nan_to_num(results[k]["val_auc"], nan=-1))
# display(threshold_filter_analysis(trades, results[best_name]["test_probs"]))


In [ ]:

# === EXPORT ===
# outdir = Path(CONFIG["output_dir"])
# outdir.mkdir(parents=True, exist_ok=True)
# qc_report.to_json(outdir / "qc_report.json", indent=2)
# trades.to_csv(outdir / "trades.csv", index=False)
# meta_ds.to_csv(outdir / "meta_label_dataset.csv")
# with open(outdir / "metrics.json", "w") as f:
#     json.dump({"overall_metrics": overall_metrics}, f, indent=2, default=str)
# with open(outdir / "run_manifest.json", "w") as f:
#     json.dump({"config": CONFIG}, f, indent=2, default=str)
# print("Exported to:", outdir.resolve())



## Catatan
Baseline ini sengaja dibuat sederhana tetapi execution-aware:
- no look-ahead,
- next-open entry,
- structural swing stop,
- SL-first tie rule,
- siap dipakai untuk audit event, session, year, dan H1 regime.
